# SoundWel Multimodal Valence Classifier — Interactive Demo

**Pattern Learning Fundamentals — Cross-Modal and Multimodal Learning Lab**

This notebook walks you through the full pipeline step by step:

1. Load and inspect the annotation file
2. Visualise both modalities for a single sample
3. Verify dataset splits
4. Check model architecture and parameter counts
5. Run a single training epoch
6. Evaluate on the test set

---
> **Before running:** make sure your data is in place and paths in `config.py` are correct.
> Run `python dataset.py` and `python model.py` from the terminal first to confirm everything loads.

## 0 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image

try:
    import librosa
    import librosa.display
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("librosa not found — waveform plot will be skipped")

from config  import Config
from dataset import SoundWelDataset, build_splits
from model   import MultimodalValenceClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 1 — Inspect the annotation file

**Lecture connection:** This is your multimodal dataset. Each row contains
pointers to TWO modalities (waveform + spectrogram) for the same pig vocalisation.

In [ ]:
df = pd.read_excel(Config.ANNOTATIONS_FILE)
print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nValence distribution:")
print(df[Config.LABEL_COL].value_counts())
df.head(5)

## 2 — Visualise both modalities for one sample

**Lecture connection (Slide 9):** The waveform is a *raw* modality (closest to sensor).
The spectrogram is one step more abstract — it converts the time-domain signal
into a time-frequency image. Both represent the same underlying vocalisation,
but with different structure → this is what makes them **heterogeneous**.

In [ ]:
# Pick the first row as an example
row = df.iloc[0]
audio_path = os.path.join(Config.AUDIO_DIR, row[Config.AUDIO_COL])
spec_path  = os.path.join(Config.SPEC_DIR,  row[Config.SPEC_COL])
label_str  = row[Config.LABEL_COL]

print(f"Audio file : {audio_path}")
print(f"Spec file  : {spec_path}")
print(f"Valence    : {label_str}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Modality 1: Waveform ──────────────────────────────────────────────────
if LIBROSA_AVAILABLE and os.path.exists(audio_path):
    y, sr = librosa.load(audio_path, sr=Config.SAMPLE_RATE, mono=True)
    times = np.linspace(0, len(y) / sr, num=len(y))
    axes[0].plot(times, y, color='steelblue', linewidth=0.6)
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(f"Modality 1: Audio Waveform  |  SR={sr} Hz", fontweight='bold')
    axes[0].set_xlim([0, len(y) / sr])
    axes[0].grid(alpha=0.3)
    print(f"Waveform: {len(y)} samples at {sr} Hz ({len(y)/sr:.3f} s)")
else:
    axes[0].text(0.5, 0.5, "librosa not available\nor file not found",
                 ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title("Modality 1: Audio Waveform", fontweight='bold')

# ── Modality 2: Spectrogram ───────────────────────────────────────────────
if os.path.exists(spec_path):
    img = Image.open(spec_path).convert("RGB")
    axes[1].imshow(img)
    axes[1].set_title(
        f"Modality 2: Spectrogram  |  {img.size[0]}×{img.size[1]} px",
        fontweight='bold'
    )
    axes[1].axis('off')
    print(f"Spectrogram: {img.size[0]}×{img.size[1]} px, mode={img.mode}")
else:
    axes[1].text(0.5, 0.5, "Spectrogram file not found",
                 ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title("Modality 2: Spectrogram", fontweight='bold')

fig.suptitle(f"Valence: {label_str}", fontsize=14, fontweight='bold',
             color='green' if label_str == 'Pos' else 'red')
plt.tight_layout()
plt.show()

**Question:** Looking at the two plots above, can you identify which dimension of
heterogeneity (Lecture Slide 8) is most obvious between the waveform and the spectrogram?
Think about: element representation, structure, information content.

## 3 — Build dataset splits and verify tensor shapes

**Lecture connection (Slide 10):** We use supervised learning — labels come from
the annotation file. The group-aware split ensures no recording session appears
in both train and test (preventing data leakage).

In [ ]:
df_train, df_val, df_test = build_splits(Config.ANNOTATIONS_FILE)

train_ds = SoundWelDataset(df_train, augment=True)
val_ds   = SoundWelDataset(df_val,   augment=False)
test_ds  = SoundWelDataset(df_test,  augment=False)

print(f"Dataset sizes — train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

# Inspect the first training sample
waveform, spectrogram, label = train_ds[0]

print(f"\nFirst training sample:")
print(f"  Waveform shape      : {tuple(waveform.shape)}   ← (channels, time_samples)")
print(f"  Spectrogram shape   : {tuple(spectrogram.shape)} ← (RGB_channels, H, W)")
print(f"  Label               : {label.item()}  (0=Neg, 1=Pos)")
print(f"  Waveform min/max    : {waveform.min():.3f} / {waveform.max():.3f}")
print(f"  Spectrogram min/max : {spectrogram.min():.3f} / {spectrogram.max():.3f}")

## 4 — Explore the class distribution

**Lecture connection:** Class imbalance is one reason we need weighted loss.
This connects to the *Noise* dimension of heterogeneity (Slide 8) — the label
distribution is uneven across recording conditions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for ax, (name, dset_df) in zip(axes, [("Train", df_train), ("Val", df_val), ("Test", df_test)]):
    counts = dset_df['label'].value_counts().sort_index()
    ax.bar(["Negative", "Positive"], counts.values,
           color=['#C0392B', '#1D6A3E'], edgecolor='white')
    ax.set_title(f"{name} set", fontweight='bold')
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.suptitle("Class distribution across splits", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Compute class weights (same logic as train.py)
neg = (df_train['label'] == 0).sum()
pos = (df_train['label'] == 1).sum()
print(f"Training — Neg: {neg}, Pos: {pos}, ratio: {neg/pos:.2f}")
print(f"Class weight for Neg: {pos/neg:.3f}  (will upweight minority class in loss)")

## 5 — Inspect the model architecture

**Lecture connection (Slide 15):** Two independent ResNet-18 encoders process
heterogeneous modalities separately (unimodal encoding), then their feature
vectors are fused via concatenation (hybrid fusion).

In [ ]:
model = MultimodalValenceClassifier(
    num_classes=Config.NUM_CLASSES,
    fusion_dim=Config.FUSION_DIM,
    mlp_hidden=Config.MLP_HIDDEN,
).to(DEVICE)

# Parameter counts per component
wave_params = sum(p.numel() for p in model.waveform_branch.parameters())
spec_params = sum(p.numel() for p in model.spec_branch.parameters())
head_params = sum(p.numel() for p in model.fusion_head.parameters())
total       = wave_params + spec_params + head_params

print("Parameter counts:")
print(f"  Waveform branch   (1D ResNet-18) : {wave_params:>10,}")
print(f"  Spectrogram branch (2D ResNet-18): {spec_params:>10,}")
print(f"  Fusion MLP head                 : {head_params:>10,}")
print(f"  ─────────────────────────────────────────")
print(f"  Total                           : {total:>10,}")

# Forward-pass shapes with dummy inputs
B = 2
dummy_wave = torch.randn(B, 1, Config.SAMPLE_LENGTH).to(DEVICE)
dummy_spec = torch.randn(B, 3, Config.SPEC_SIZE, Config.SPEC_SIZE).to(DEVICE)

with torch.no_grad():
    feat_w = model.waveform_branch(dummy_wave)
    feat_s = model.spec_branch(dummy_spec)
    fused  = torch.cat([feat_w, feat_s], dim=1)
    logits = model(dummy_wave, dummy_spec)

print(f"\nForward-pass shapes (batch size = {B}):")
print(f"  Waveform input         : {tuple(dummy_wave.shape)}")
print(f"  Waveform features      : {tuple(feat_w.shape)}  ← 512-d per sample")
print(f"  Spectrogram input      : {tuple(dummy_spec.shape)}")
print(f"  Spectrogram features   : {tuple(feat_s.shape)}  ← 512-d per sample")
print(f"  Concatenated features  : {tuple(fused.shape)}  ← 1024-d per sample")
print(f"  Output logits          : {tuple(logits.shape)}  ← 2 scores per sample")

## 6 — Visualise what each branch receives

Show a real waveform tensor and a real spectrogram tensor from the first batch.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=False, num_workers=0)
waveforms, spectrograms, labels = next(iter(train_loader))

print(f"Batch shapes — waveforms: {tuple(waveforms.shape)}, spectrograms: {tuple(spectrograms.shape)}")
print(f"Labels: {labels.tolist()}")

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i in range(4):
    # Waveform (top row)
    wav = waveforms[i, 0].numpy()
    axes[0, i].plot(wav, color='steelblue', linewidth=0.4)
    axes[0, i].set_title(f"Waveform — {'Pos' if labels[i]==1 else 'Neg'}",
                          fontweight='bold',
                          color='green' if labels[i]==1 else 'red')
    axes[0, i].set_xlabel("Sample index")
    axes[0, i].grid(alpha=0.3)

    # Spectrogram (bottom row) — de-normalise for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = spectrograms[i] * std + mean          # undo ImageNet normalisation
    img  = img.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"Spectrogram — {'Pos' if labels[i]==1 else 'Neg'}",
                          fontweight='bold',
                          color='green' if labels[i]==1 else 'red')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel("Modality 1\nAmplitude", fontsize=11)
axes[1, 0].set_ylabel("Modality 2\nSpectrogram", fontsize=11)
plt.suptitle("First batch: both modalities side by side", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7 — Train for a few epochs (quick run)

This cell runs a short training loop (3 epochs) so you can see the
loss curve without waiting for the full 50-epoch run.

For the full training run, use `python train.py` from the terminal.

In [ ]:
from sklearn.metrics import accuracy_score
from evaluate import evaluate

# Fresh model
model = MultimodalValenceClassifier().to(DEVICE)

# Weighted loss
neg_count = (df_train['label'] == 0).sum()
pos_count = (df_train['label'] == 1).sum()
weights = torch.tensor([pos_count / neg_count, 1.0], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.LEARNING_RATE,
                             weight_decay=Config.WEIGHT_DECAY)

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                          shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=0)

QUICK_EPOCHS = 3
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
print("─" * 55)

for epoch in range(1, QUICK_EPOCHS + 1):
    model.train()
    total_loss, all_labels, all_preds = 0.0, [], []

    for waveforms, spectrograms, labels_batch in train_loader:
        waveforms    = waveforms.to(DEVICE)
        spectrograms = spectrograms.to(DEVICE)
        labels_batch = labels_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(waveforms, spectrograms)
        loss   = criterion(logits, labels_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels_batch)
        all_labels.extend(labels_batch.cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())

    t_loss = total_loss / len(train_loader.dataset)
    t_acc  = accuracy_score(all_labels, all_preds)
    v_loss, v_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    print(f"{epoch:>5}  {t_loss:>10.4f}  {t_acc:>9.4f}  {v_loss:>8.4f}  {v_acc:>7.4f}")

print("\nQuick training done. Run train.py for the full 50-epoch run.")

## 8 — Plot the (short) training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, QUICK_EPOCHS + 1)

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   's--', label='Val',   color='tomato')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves (quick run)", fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['train_acc'], 'o-', label='Train', color='steelblue')
axes[1].plot(epochs, history['val_acc'],   's--', label='Val',   color='tomato')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy curves (quick run)", fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFor the full training run with confusion matrix and AUC, run:")
print("  python train.py")

## 9 — Ablation: test waveform-only and spectrogram-only

**Lecture connection (Slide 6 — Interconnected modalities):** If the waveform alone
and the spectrogram alone both perform worse than the combined model, the two
modalities exhibit **Enhancement** — their combination is stronger than either alone.

In [ ]:
# ── Waveform-only ablation ─────────────────────────────────────────────────
class WaveformOnlyClassifier(nn.Module):
    """Ablation: waveform branch only (no spectrogram)."""
    def __init__(self):
        super().__init__()
        from model import ResNet18_1D
        self.branch = ResNet18_1D(in_channels=1)
        self.head   = nn.Linear(Config.FUSION_DIM, Config.NUM_CLASSES)

    def forward(self, waveform, spectrogram):
        # spectrogram argument is accepted but ignored
        return self.head(self.branch(waveform))

# ── Spectrogram-only ablation ──────────────────────────────────────────────
class SpectrogramOnlyClassifier(nn.Module):
    """Ablation: spectrogram branch only (no waveform)."""
    def __init__(self):
        super().__init__()
        from model import ResNet18_2D
        self.branch = ResNet18_2D(pretrained=True)
        self.head   = nn.Linear(Config.FUSION_DIM, Config.NUM_CLASSES)

    def forward(self, waveform, spectrogram):
        # waveform argument is accepted but ignored
        return self.head(self.branch(spectrogram))


def quick_train_eval(model_cls, label):
    """Train for QUICK_EPOCHS and return val accuracy."""
    m = model_cls().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=Config.LEARNING_RATE)
    for _ in range(QUICK_EPOCHS):
        m.train()
        for wav, spec, lbl in train_loader:
            wav, spec, lbl = wav.to(DEVICE), spec.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()
            loss = criterion(m(wav, spec), lbl)
            loss.backward()
            opt.step()
    _, acc, _, _ = evaluate(m, val_loader, criterion, DEVICE)
    print(f"  {label:<30} val accuracy: {acc:.4f}")
    return acc

print(f"Ablation study ({QUICK_EPOCHS} epochs each):")
acc_wave = quick_train_eval(WaveformOnlyClassifier,    "Waveform only")
acc_spec = quick_train_eval(SpectrogramOnlyClassifier, "Spectrogram only")
acc_both = history['val_acc'][-1]  # from the multimodal run above
print(f"  {'Multimodal (both)':<30} val accuracy: {acc_both:.4f}")

print(f"\nConclusion after {QUICK_EPOCHS} epochs:")
if acc_both > max(acc_wave, acc_spec):
    print("  ✓ Multimodal outperforms both unimodal baselines → Enhancement")
else:
    print("  ✗ Multimodal does not yet outperform unimodal (try more epochs!)")

## 10 — Summary

| Step | What you did | Lecture concept |
|------|-------------|----------------|
| Annotaton inspection | Identified two modality columns | Heterogeneous modalities (Slide 4) |
| Visualisation | Saw raw waveform vs spectrogram | Unimodal representations (Slide 9) |
| Splits | Group-aware train/val/test | Supervised learning (Slide 10) |
| Architecture | Two ResNet-18 + concat + MLP | Hybrid fusion (Slide 13) |
| Training | Cross-entropy + Adam + LR scheduler | Learning loop |
| Ablation | Waveform-only vs spectrogram-only | Interconnected modalities (Slide 6) |

---

For the **full training run** with all 50 epochs, best-model checkpointing,
and a confusion matrix, run from the terminal:

```bash
python train.py
```

See `LAB_INSTRUCTIONS.md` for the complete list of experiments and report questions.